In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import random_split, Subset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torchvision.models import resnet18
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

1


Загрузка, преобразование и разделение датасета на train, валидацию и test

In [2]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset_path = './archive/archive/simpsons_dataset'
train_dataset = ImageFolder(root=dataset_path, transform=train_transform)
test_dataset = ImageFolder(root=dataset_path, transform=test_transform)

print(f"Найдено классов: {len(train_dataset.classes)}")
print(f"Всего изображений: {len(train_dataset)}")

# Разделение на train/valid/test
train_size = int(0.6 * len(train_dataset))
valid_size = int(0.2 * len(train_dataset))
test_size = len(train_dataset) - train_size - valid_size

indices = list(range(len(train_dataset)))
train_indices, valid_indices, test_indices = random_split(indices, [train_size, valid_size, test_size])

train_subset = Subset(train_dataset, train_indices)
valid_subset = Subset(test_dataset, valid_indices)
test_subset = Subset(test_dataset, test_indices)

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
valid_loader = DataLoader(valid_subset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_subset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

Найдено классов: 42
Всего изображений: 20933


<h8>Модель для обучения<h8>

In [4]:
class SimpsonModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        backbone = resnet18(pretrained=True)
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        self.bn = nn.BatchNorm1d(512)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(512, num_classes)
        
    def forward(self, x):
        x = self.backbone(x)
        x = x.view(x.size(0), -1)
        x = self.bn(x)
        x = self.dropout(x)
        return self.classifier(x)

<h8>Метрика<h8>

In [5]:
def calculate_f1(labels, predictions, num_classes, average='weighted'):
    labels_np = labels.cpu().numpy()
    predictions_np = predictions.cpu().numpy()
    
    f1 = f1_score(labels_np, predictions_np, average=average, zero_division=0)
    precision = precision_score(labels_np, predictions_np, average=average, zero_division=0)
    recall = recall_score(labels_np, predictions_np, average=average, zero_division=0)
    
    return f1, precision, recall

<h8>Обучение модели<h8>

In [6]:
def train_model(model, train_loader, valid_loader, optimizer, num_epochs, num_classes):
    best_valid_f1 = 0
    best_valid_precision = 0
    best_valid_recall = 0

    for epoch in range(num_epochs):
        print(f'\nЭпоха {epoch+1}/{num_epochs}')

        # Обучение
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        all_train_labels = []
        all_train_preds = []

        for images, labels in train_loader:
            images, labels = images.cuda(), labels.cuda()

            optimizer.zero_grad()
            logits = model(images)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = logits.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()

            all_train_labels.extend(labels.cpu())
            all_train_preds.extend(predicted.cpu())

        train_loss /= len(train_loader)
        train_acc = 100. * train_correct / train_total
        train_f1, _, _ = calculate_f1(torch.tensor(all_train_labels), torch.tensor(all_train_preds), num_classes)

        # Валидация
        model.eval()
        valid_loss = 0
        valid_correct = 0
        valid_total = 0
        all_valid_labels = []
        all_valid_preds = []

        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.cuda(), labels.cuda()
                logits = model(images)
                loss = F.cross_entropy(logits, labels)

                valid_loss += loss.item()
                _, predicted = logits.max(1)
                valid_total += labels.size(0)
                valid_correct += predicted.eq(labels).sum().item()

                all_valid_labels.extend(labels.cpu())
                all_valid_preds.extend(predicted.cpu())

        valid_loss /= len(valid_loader)
        valid_acc = 100. * valid_correct / valid_total

        valid_f1, valid_precision, valid_recall = calculate_f1(torch.tensor(all_valid_labels), torch.tensor(all_valid_preds), num_classes, 
                                                               average='weighted')

        print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, Train F1: {train_f1:.4f}')
        print(f'Valid Loss: {valid_loss:.4f}, Valid Acc: {valid_acc:.2f}%')
        print(f'Valid F1: {valid_f1:.4f}, Precision: {valid_precision:.4f}, Recall: {valid_recall:.4f}')

        if valid_f1 > best_valid_f1:
            best_valid_f1 = valid_f1
            torch.save(model.state_dict(), 'best_model.pth')

        if valid_precision > best_valid_precision:
            best_valid_precision = valid_precision
            torch.save(model.state_dict(), 'best_model_precision.pth')

        if valid_recall > best_valid_recall:
            best_valid_recall = valid_recall
            torch.save(model.state_dict(), 'best_model_recall.pth')

<h8>Тест<h8>

In [7]:
def test_model(model, test_loader, num_classes):
    print("Test")

    model.load_state_dict(torch.load('best_model.pth'))
    model.eval()

    test_loss = 0
    test_correct = 0
    test_total = 0
    all_test_labels = []
    all_test_preds = []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.cuda(), labels.cuda()
            logits = model(images)
            loss = F.cross_entropy(logits, labels)

            test_loss += loss.item()
            _, predicted = logits.max(1)
            test_total += labels.size(0)
            test_correct += predicted.eq(labels).sum().item()

            all_test_labels.extend(labels.cpu())
            all_test_preds.extend(predicted.cpu())

    test_loss /= len(test_loader)
    test_acc = 100. * test_correct / test_total

    test_f1_weighted, test_precision, test_recall = calculate_f1(torch.tensor(all_test_labels), torch.tensor(all_test_preds), num_classes,
                                                                 average='weighted')
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.2f}%")
    print(f"Test F1 (weighted): {test_f1_weighted:.4f}")
    print(f"Precision: {test_precision:.4f}")
    print(f"Recall: {test_recall:.4f}")

In [8]:
num_classes = len(train_dataset.classes)
model = SimpsonModel(num_classes=num_classes).cuda()

optimizer = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-3)

train_model(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    optimizer=optimizer,
    num_epochs=6,
    num_classes=num_classes
)

C:\Users\user\programs\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\user\programs\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Эпоха 1/20
Train Loss: 1.6139, Train Acc: 64.34%, Train F1: 0.6475
Valid Loss: 0.5843, Valid Acc: 86.74%
Valid F1: 0.8415, Precision: 0.8365, Recall: 0.8674

Эпоха 2/20
Train Loss: 0.4676, Train Acc: 89.70%, Train F1: 0.8813
Valid Loss: 0.3510, Valid Acc: 92.16%
Valid F1: 0.9106, Precision: 0.9079, Recall: 0.9216

Эпоха 3/20
Train Loss: 0.2735, Train Acc: 94.04%, Train F1: 0.9330
Valid Loss: 0.2597, Valid Acc: 93.91%
Valid F1: 0.9329, Precision: 0.9353, Recall: 0.9391

Эпоха 4/20
Train Loss: 0.1785, Train Acc: 96.35%, Train F1: 0.9609
Valid Loss: 0.2182, Valid Acc: 95.22%
Valid F1: 0.9492, Precision: 0.9508, Recall: 0.9522

Эпоха 5/20
Train Loss: 0.1237, Train Acc: 97.75%, Train F1: 0.9762
Valid Loss: 0.1829, Valid Acc: 95.96%
Valid F1: 0.9576, Precision: 0.9596, Recall: 0.9596

Эпоха 6/20
Train Loss: 0.0818, Train Acc: 98.83%, Train F1: 0.9877
Valid Loss: 0.1643, Valid Acc: 96.49%
Valid F1: 0.9634, Precision: 0.9643, Recall: 0.9649

Эпоха 7/20


KeyboardInterrupt: 

In [14]:
test_model(model=model, test_loader=test_loader, num_classes=num_classes)

Test


C:\Users\user\AppData\Local\Temp\ipykernel_153632\1841898317.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pth'))


Test Loss: 0.1660
Test Accuracy: 96.66%
Test F1 (weighted): 0.9658
Precision: 0.9665
Recall: 0.9666
